# 18.1 教師能提供什麼學習訊號？


手上已有一筆原始資料：問題「2+2=?」，標準答案「4」。小模型可以直接用這份資料學習，提高答案4的機率。現在請一個已訓練好的教師來指導同一個學生：教師若只交出文字「4」，學生收到的答案仍然是4；教師若交出候選 `[4,3,100]` 的比例 `[0.7,0.2,0.1]`，學生還能看見第一名以外的偏好。問題沒換，收到的學習目標卻可能不同。

蒸餾（distillation）指讓學生透過教師提供的訊號學習。原始資料和標準答案都在手上，仍可以加入教師指導；不必等到原資料拿不到才使用蒸餾。教師也不一定是真值，候選比例只是它在當下前文裡的模型偏好，不是宣稱2+2有10%的機會等於100。

下圖把同一題分成三條可選的訓練路。比較時，三條路使用同一學生架構、同一份起始權重；第一條學原答案，第二條學教師交出的相同答案，第三條另外取得候選比例。箭頭表示同一題可以交給哪條路，不表示三條路依次訓練。這裡的比例是人工示例，後面會算出它如何改變學生的調整方向。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/p7-18-three-targets.svg")))

[下一字分數](https://birdhackor.github.io/tiny-perceptron-vlm/4.6.html)常叫logits，softmax把它們轉成總和為一的機率分布；[SFT](https://birdhackor.github.io/tiny-perceptron-vlm/7.11.html)則用回答文字作目標，讓模型增加每個示範答案token的機率。只能看到教師生成文字的方式常叫黑盒蒸餾；可讀取教師分布的方式常叫白盒蒸餾。這些名字描述能拿到的訊號，不表示教師是不是可靠。

沿用問題「2+2=?」和候選 `[4,3,100]`，只把教師比例換成另一份 `[0.4,0.35,0.25]`。原本的 `[0.7,0.2,0.1]` 和這份比例，第一名都是4。只取得「4」無法分辨這兩種情況：它沒有記錄第二名與第一名的差距，也沒有記錄100的相對位置。


In [ ]:
import torch

candidates = ["4", "3", "100"]
probabilities = [torch.tensor([0.7, 0.2, 0.1]), torch.tensor([0.4, 0.35, 0.25])]
for prob in probabilities:
    answer = candidates[prob.argmax().item()]
    print("回答", answer, "分布", prob.tolist())

輸入是人工指定的兩個分布，用來隔離資訊差別，不是呼叫真的教師。這裡把三個答案當成三個候選；真實切詞器可能把100切成多個token，不能把這三欄當成實際byte詞表。`argmax`選最大機率索引，再查候選文字，兩行回答都應為4；分布欄卻不同。實際回答是一串token，黑盒資料保存整串文字；白盒通常要保存或即時讀取每個有效回答位置的一整列詞表分布，資料量和計算需求可能更大。

取得文字後，可由學生自己的tokenizer重新切分，所以黑盒方式能搭配不同詞表。逐欄對機率則必須確保兩邊同一欄代表同一個token，這是[詞表對齊](https://birdhackor.github.io/tiny-perceptron-vlm/18.6.html)會處理的問題。白盒能提供更多候選關係，也可能把教師的錯誤偏好一起交給學生；多一份資訊不等於保證更好。

練習換成分類任務，候選是 `[貓,狗,鳥]`。兩份人工教師比例分別是 `[0.8,0.15,0.05]` 和 `[0.8,0.05,0.15]`。若資料只保存第一名「貓」，能判斷教師認為狗還是鳥比較接近嗎？先說理由，再把候選和比例換進程式核對。兩份第一名和第一名的機率都相同，次要候選的順序仍不同；留下的文字不足以還原這份關係。

<details>
<summary>選讀：訊號分類、教師來源與方法範圍</summary>

[Hinton等人原文第1節](https://arxiv.org/abs/1503.02531v1)明說蒸餾可使用原訓練集，也可用另一份轉移資料；有原標籤時，還可以同時學真實答案與教師軟目標。教師比例能傳遞已學出的候選關係，不等於新增真值，也不保證學生進步。

[MiniLLM原文的導論](https://arxiv.org/abs/2306.08543v6)也按可取得的文字或分布區分這兩類訊號。本章的正式對照使用普通CE與固定前文上的forward KL，沒有採用該論文的reverse KL或policy-gradient方法。教師來源則是[直接SFT](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.4)、[條件式風格](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.5)與[MoE的固定top-2支線](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.8)，都載入實際訓練權重。它們的完整檔案SHA、架構與步數保存在[蒸餾實報的`teacher_provenance`](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/distillation.json)。

</details>
